# CO₂ factor: combined historical data EDA

Explore **every file in `data/raw/co2` together**: schema, temporal coverage, missing observations, duplicates, unusual values, and seasonal patterns. Compare suitability with `docs/initial_assignment.md` (a 168-hour forecast using renewable production forecasts).

This is descriptive analysis, not model evaluation. Preserve raw values; flag possible anomalies without removing or imputing them. UTC defines the regular hourly grid; Europe/Amsterdam defines calendar profiles. Statistical outliers are not automatically errors.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "docs/initial_assignment.md").exists())
DATA_DIR = ROOT / "data/raw/co2"
TARGET = "emissionfactor (kg CO2/kWh)"

plt.rcParams.update({"figure.figsize": (12, 4), "axes.grid": True, "grid.alpha": 0.25})

## 1. Load every source file

The directory currently contains CSV exports. Fail visibly if an unexpected file format or schema appears, rather than silently omitting it. Keep source filenames to inspect boundaries and provenance. `validfrom` is the interval start and `validto` is its end; both headers explicitly specify UTC.

In [ ]:
files = sorted(p for p in DATA_DIR.iterdir() if p.is_file())
assert files, "No input files found"
assert all(p.suffix.lower() == ".csv" for p in files), "Review unsupported source files"

frames = [pd.read_csv(p) for p in files]
assert all(frame.columns.equals(frames[0].columns) for frame in frames), "Source schemas differ"

inventory = pd.DataFrame({"file": [p.name for p in files], "bytes": [p.stat().st_size for p in files], "rows": [len(f) for f in frames]})
df = pd.concat([f.assign(source_file=p.name) for p, f in zip(files, frames)], ignore_index=True)
raw_columns = list(frames[0].columns)

for col in ["validfrom (UTC)", "validto (UTC)"]:
    df[col] = pd.to_datetime(df[col], utc=True, errors="coerce")
df = df.sort_values("validfrom (UTC)").reset_index(drop=True)

display(inventory)

In [ ]:
display(df.head())

In [ ]:
display(pd.DataFrame({"dtype": df.dtypes.astype(str), "unique_values": df.nunique(dropna=False)}))

In [ ]:
print(f"Total: {len(df):,} rows, {len(raw_columns)} original columns")

`emissionfactor (kg CO2/kWh)` is the requested carbon-intensity target; multiply by 1,000 for g CO₂/kWh. `emission (kg CO2)` is an absolute mass and is not the target. Capacity and volume have distinct physical units even when numerically equal over one-hour intervals. The meaning and denominator of `percentage` are not documented in these exports; do not interpret it as renewable share.

Inspect constant metadata to confirm geography, activity, resolution, and observation classification. `Current` alone does not establish release times or revision history.

In [ ]:
metadata_columns = ["point", "type", "granularity", "timezone", "activity", "classification"]
display(pd.DataFrame({c: [df[c].unique().tolist()] for c in metadata_columns}).T.rename(columns={0: "observed_values"}))

In [ ]:
numeric_columns = df[raw_columns].select_dtypes(include="number").columns.tolist()
display(df[numeric_columns].describe(percentiles=[.01, .25, .5, .75, .99]).T)

In [ ]:
display(df.groupby("source_file").agg(rows=(TARGET, "size"), start_utc=("validfrom (UTC)", "min"), end_utc_exclusive=("validto (UTC)", "max"), target_mean=(TARGET, "mean")))

## 2. Missing values, duplicate rows, and missing hours

A file can have no null cells while still omitting entire hours. Check both, including cross-file boundaries. Do not resolve duplicate timestamps silently. Coverage is measured only between the first and last observed timestamps; this cannot establish completeness outside that range.

In [ ]:
display(pd.DataFrame({"missing": df.isna().sum(), "missing_pct": df.isna().mean() * 100}))

In [ ]:
start = df["validfrom (UTC)"]
end = df["validto (UTC)"]
duplicate_times = start.duplicated(keep=False)
expected = pd.date_range(start.min(), start.max(), freq="h")
missing_hours = expected.difference(pd.DatetimeIndex(start.dropna()))
durations = (end - start).dt.total_seconds() / 3600
checks = pd.Series({
    "duplicate_raw_rows": df.duplicated(raw_columns).sum(),
    "rows_with_duplicate_start": duplicate_times.sum(),
    "invalid_start_or_end": (start.isna() | end.isna()).sum(),
    "non_hour_intervals": durations.ne(1).sum(),
    "off_hour_starts": start.ne(start.dt.floor("h")).sum(),
    "missing_hours_inside_coverage": len(missing_hours),
    "non_contiguous_adjacent_intervals": (start.iloc[1:].reset_index(drop=True) != end.iloc[:-1].reset_index(drop=True)).sum(),
})
display(checks.to_frame("count"))

In [ ]:
display(pd.DataFrame({"missing_hour_utc": missing_hours}).head(20))

In [ ]:
display(df.loc[duplicate_times].head(10))

In [ ]:
assert not duplicate_times.any() and start.notna().all(), "Resolve timestamp issues before time-series analysis"
s = df.set_index("validfrom (UTC)")[TARGET].reindex(expected)
s.index.name = "time_utc"
local = s.tz_convert("Europe/Amsterdam")
print(f"Local interval coverage: {start.min().tz_convert("Europe/Amsterdam")} to {end.max().tz_convert("Europe/Amsterdam")} (end exclusive)")

## 3. Plausibility and internal consistency

Check finite values, negative quantities, capacity × interval length versus energy, and emission mass / energy versus the supplied factor. Differences may reflect aggregation or rounding and require investigation rather than automatic correction. These checks cannot verify the provider's emissions accounting methodology.

In [ ]:
display(pd.DataFrame({
    "non_finite": (~np.isfinite(df[numeric_columns])).sum(),
    "negative": df[numeric_columns].lt(0).sum(),
    "zero": df[numeric_columns].eq(0).sum(),
}))

In [ ]:
ratio = df["emission (kg CO2)"].div(df["volume (kWh)"].replace(0, np.nan))
factor_error = df[TARGET] - ratio
energy_error = df["volume (kWh)"] - df["capacity (kW)"] * durations
display(pd.DataFrame({"factor_minus_mass_over_energy": factor_error, "volume_minus_capacity_times_hours": energy_error}).describe().T)

In [ ]:
print(f"Max absolute factor discrepancy: {factor_error.abs().max():.8f} kg CO2/kWh")

## 4. Distribution and possible outliers

Use the conventional 1.5 × IQR rule as a descriptive flag for each numeric field, not a physical validity threshold. Also flag unusually large absolute hour-to-hour target changes using the 99th percentile. These relative rules can flag valid system events and may miss sustained abnormal regimes; retain every value.

In [ ]:
q1, q3 = df[numeric_columns].quantile(.25), df[numeric_columns].quantile(.75)
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
flags = df[numeric_columns].lt(lower) | df[numeric_columns].gt(upper)
display(pd.DataFrame({"lower_fence": lower, "upper_fence": upper, "flagged_rows": flags.sum(), "flagged_pct": flags.mean() * 100}))

In [ ]:
changes = s.diff().abs()
jump_threshold = changes.quantile(.99)
jump_flags = changes.gt(jump_threshold)
display(pd.DataFrame({"factor": s, "absolute_hourly_change": changes}).nlargest(10, "absolute_hourly_change"))

In [ ]:
fig, ax = plt.subplots()
ax.hist(s.dropna() * 1000, bins=60)
ax.set(xlabel="CO₂ factor (g CO₂/kWh)", ylabel="Hours", title="Combined target distribution")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.boxplot([g.dropna().to_numpy() * 1000 for _, g in local.groupby(local.index.year)], tick_labels=[str(y) for y in sorted(local.index.year.unique())])
ax.set(xlabel="Local year (2026 partial)", ylabel="CO₂ factor (g CO₂/kWh)", title="Distribution by year")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.hist(changes.dropna() * 1000, bins=60)
ax.axvline(jump_threshold * 1000, color="red", label="99th percentile")
ax.set(xlabel="Absolute hourly change (g CO₂/kWh)", ylabel="Hours", title="Changes between adjacent hours")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Time-series views

Plot hourly observations with daily and monthly means, the latest 28 days, and flagged jumps. Monthly means near dataset boundaries may cover partial months. Monthly coverage counts below make this visible.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(s.index, s * 1000, alpha=.25, linewidth=.4, label="Hourly")
ax.plot(s.resample("D").mean() * 1000, linewidth=.8, label="Daily mean")
ax.plot(s.resample("MS").mean() * 1000, linewidth=2, label="Monthly mean")
ax.scatter(s.index[jump_flags], s[jump_flags] * 1000, s=8, color="red", label="Large hourly change")
ax.set(title="Combined history", ylabel="CO₂ factor (g CO₂/kWh)", xlabel="UTC")
ax.legend(ncol=4)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
recent = s.loc[s.index.max() - pd.Timedelta(days=28):]
ax.plot(recent.index, recent * 1000)
ax.set(title="Latest 28 days", ylabel="CO₂ factor (g CO₂/kWh)", xlabel="UTC")
plt.tight_layout()
plt.show()

In [ ]:
display(local.resample("MS").agg(["count", "mean", "min", "max"]).rename_axis("local_month"))

## 6. Daily, weekly, and annual seasonality

Calendar profiles use Dutch local time, including daylight saving transitions. UTC stays regular; local spring days have 23 hours and autumn days 25. Compare years separately because 2026 is incomplete. Counts expose unequal coverage. Profiles show association, not a causal or stationary seasonal effect; fewer than three complete annual cycles limit annual conclusions.

In [ ]:
calendar = local.rename("factor").to_frame()
calendar["year"] = calendar.index.year
calendar["month"] = calendar.index.month
calendar["weekday"] = calendar.index.dayofweek
calendar["hour"] = calendar.index.hour
fig, ax = plt.subplots()

for year, group in calendar.groupby("year"):
    (group.groupby("hour")["factor"].mean() * 1000).plot(ax=ax, marker="o", label=str(year))
ax.set(xlabel="Local hour", ylabel="Mean CO₂ factor (g CO₂/kWh)", title="Daily profile")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()
for year, group in calendar.groupby("year"):
    (group.groupby("weekday")["factor"].mean() * 1000).plot(ax=ax, marker="o", label=str(year))
ax.set(xlabel="Local weekday", ylabel="Mean CO₂ factor (g CO₂/kWh)", title="Weekly profile (Monday = 0)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()
for year, group in calendar.groupby("year"):
    (group.groupby("month")["factor"].mean() * 1000).plot(ax=ax, marker="o", label=str(year))
ax.set(xlabel="Local month", ylabel="Mean CO₂ factor (g CO₂/kWh)", title="Annual profile (2026 partial)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
display(calendar.groupby(["year", "month"])["factor"].agg(["count", "mean"]).unstack("year"))

In [ ]:
heatmap = calendar.pivot_table(index="weekday", columns="hour", values="factor", aggfunc="mean") * 1000
fig, ax = plt.subplots(figsize=(12, 3))
im = ax.imshow(heatmap, aspect="auto", origin="lower")
ax.set(xticks=range(24), yticks=range(7), yticklabels=["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"], xlabel="Local hour", title="Mean CO₂ factor by weekday and hour")
fig.colorbar(im, ax=ax, label="g CO₂/kWh")
plt.tight_layout()
plt.show()

### Autocorrelation on a regular hourly grid

Compare raw autocorrelation with autocorrelation after subtracting a centered 30-day rolling mean to reduce slow variation. The centered smoother is for retrospective EDA only and would leak future information if used as a forecasting feature. Correlations use available aligned pairs without filling gaps. Peaks around 24 and 168 hours support daily and weekly recurrence but are not a formal seasonality test or evidence of forecast accuracy.

In [ ]:
detrended = s - s.rolling(24 * 30, center=True, min_periods=24 * 15).mean()
lags = np.arange(1, 24 * 14 + 1)
acf = pd.DataFrame({"raw": [s.autocorr(int(lag)) for lag in lags], "detrended": [detrended.autocorr(int(lag)) for lag in lags]}, index=lags)
ax = acf.plot(figsize=(12, 4))

for lag in [24, 168]:
    ax.axvline(lag, color="grey", linestyle="--", alpha=.7)
ax.set(xlabel="Lag (hours)", ylabel="Pearson autocorrelation", title="Hourly autocorrelation: daily and weekly recurrence")
plt.show()

In [ ]:
display(pd.DataFrame({"raw_correlation": [s.autocorr(lag) for lag in [1, 24, 168]], "detrended_correlation": [detrended.autocorr(lag) for lag in [1, 24, 168]], "available_pairs": [(s.notna() & s.shift(lag).notna()).sum() for lag in [1, 24, 168]]}, index=pd.Index([1, 24, 168], name="lag_hours")))

## 7. Findings and next steps

The following summary is calculated from the loaded files. Inspect the seasonal charts together with the correlations: different weather and generation mixes can change seasonal amplitudes across years.

In [ ]:
display(Markdown(f"""
- Loaded **{len(files)} files** and **{len(df):,} hourly rows**. UTC start coverage: **{s.index.min()} – {s.index.max()}**.
- Missing raw cells: **{df[raw_columns].isna().sum().sum():,}**; missing interior hours: **{len(missing_hours):,}**; duplicate-start rows: **{duplicate_times.sum():,}**.
- Target range: **{s.min() * 1000:.2f}–{s.max() * 1000:.2f} g CO₂/kWh**; mean **{s.mean() * 1000:.2f}**.
- Maximum absolute difference between the published factor and reported mass / energy: **{factor_error.abs().max() * 1000:.3f} g CO₂/kWh**. Preserve the published target and investigate the accounting or aggregation difference.
- Target IQR flags: **{flags[TARGET].sum():,}**; large hourly changes: **{jump_flags.sum():,}** above **{jump_threshold * 1000:.2f} g CO₂/kWh**. Flags were retained.
- Raw autocorrelation: **{s.autocorr(24):.3f}** at 24 hours and **{s.autocorr(168):.3f}** at 168 hours; detrended: **{detrended.autocorr(24):.3f}** and **{detrended.autocorr(168):.3f}**. Assess recurrence alongside local calendar profiles rather than treating these numbers as proof of stable seasonality.
- **2026 is incomplete**. Compare matching calendar periods before attributing annual differences to changes in the system.

Next: use the published total-mix factor as the target, renewable interval-energy histories as the core inputs, and calendar features plus target history available at each forecast origin. Archived forecasts are unavailable; the agreed initial experiment assumes known future actual generation and excludes generation forecast error. Investigate the factor versus mass/energy discrepancy without replacing the target automatically. Evaluate chronological 168-hour forecasts against persistence and repeated daily/weekly profiles; fit preprocessing on training data only. See [the consolidated EDA report](../eda_report.md) for feature selection and preprocessing decisions.
"""))